In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

In [8]:
DATA_PATH = 'archive/dataset/dataset/Annual_P_L_1_final.csv'
OUTPUT_PATH = 'output/opm_classification/'

In [9]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

Loaded: 4668 rows × 58 columns


In [10]:
print(f"\nOPM Statistics:")
print(df['OPM'].describe())


OPM Statistics:
count      4367.000000
mean        -89.535244
std        2264.934140
min     -102800.000000
25%           2.780000
50%           9.910000
75%          19.280000
max        3094.120000
Name: OPM, dtype: float64


In [11]:
print(f"\n🔍 OPM Distribution:")
print(f"   Missing values: {df['OPM'].isna().sum()} ({df['OPM'].isna().sum()/len(df)*100:.1f}%)")
print(f"   Min: {df['OPM'].min():.2f}%")
print(f"   Max: {df['OPM'].max():.2f}%")
print(f"   Mean: {df['OPM'].mean():.2f}%")
print(f"   Median: {df['OPM'].median():.2f}%")


🔍 OPM Distribution:
   Missing values: 301 (6.4%)
   Min: -102800.00%
   Max: 3094.12%
   Mean: -89.54%
   Median: 9.91%


In [12]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df['OPM'].hist(bins=100, edgecolor='black')
plt.xlabel('OPM (%)')
plt.ylabel('Frequency')
plt.title('OPM Distribution (Raw)')
plt.axvline(x=0, color='red', linestyle='--', label='Zero line')
plt.axvline(x=10, color='orange', linestyle='--', label='10%')
plt.axvline(x=20, color='green', linestyle='--', label='20%')
plt.legend()

plt.subplot(1, 2, 2)
df_opm_clean = df[(df['OPM'] >= -50) & (df['OPM'] <= 50)]
df_opm_clean['OPM'].hist(bins=50, edgecolor='black')
plt.xlabel('OPM (%)')
plt.ylabel('Frequency')
plt.title('OPM Distribution (Filtered -50% to 50%)')
plt.axvline(x=0, color='red', linestyle='--', label='Zero line')
plt.axvline(x=10, color='orange', linestyle='--', label='10%')
plt.axvline(x=20, color='green', linestyle='--', label='20%')
plt.legend()

plt.tight_layout()
os.makedirs(OUTPUT_PATH, exist_ok=True)
plt.savefig(f'{OUTPUT_PATH}01_opm_distribution_raw.png', dpi=300, bbox_inches='tight')
print(f"\n   💾 Saved: {OUTPUT_PATH}01_opm_distribution_raw.png")
plt.close()


   💾 Saved: output/opm_classification/01_opm_distribution_raw.png


In [13]:
print("\n🧹 STEP 1: Remove obvious data errors...")

df_filtered = df[
    (df['OPM'] > -100) &   # No company loses 100x more than it earns
    (df['OPM'] < 200)      # No company has 200% OPM
].copy()

n_removed_step1 = len(df) - len(df_filtered)
pct_removed_step1 = n_removed_step1 / len(df) * 100

print(f"   Removed {n_removed_step1} obvious errors ({pct_removed_step1:.2f}%)")
print(f"   Remaining: {len(df_filtered)} samples")


🧹 STEP 1: Remove obvious data errors...
   Removed 501 obvious errors (10.73%)
   Remaining: 4167 samples


In [14]:
print("\nSTEP 2: Winsorization...")
lower_percentile = 1
upper_percentile = 99

opm_lower = df_filtered['OPM'].quantile(lower_percentile/100)
opm_upper = df_filtered['OPM'].quantile(upper_percentile/100)

print(f"   {lower_percentile}th percentile: {opm_lower:.2f}%")
print(f"   {upper_percentile}th percentile: {opm_upper:.2f}%")

n_lower = (df_filtered['OPM'] < opm_lower).sum()
n_upper = (df_filtered['OPM'] > opm_upper).sum()
print(f"   Values to cap (lower): {n_lower}")
print(f"   Values to cap (upper): {n_upper}")

df_filtered['OPM_clean'] = df_filtered['OPM'].clip(opm_lower, opm_upper)


STEP 2: Winsorization...
   1th percentile: -61.06%
   99th percentile: 92.62%
   Values to cap (lower): 42
   Values to cap (upper): 42


In [15]:
print("\nSTEP 3: Apply business rules...")
original_lower = df_filtered['OPM_clean'].min()
original_upper = df_filtered['OPM_clean'].max()

BUSINESS_MIN = -50
BUSINESS_MAX = 100

if opm_lower < BUSINESS_MIN:
    print(f"   ⚠️  Lower bound ({opm_lower:.2f}%) too extreme, capping at {BUSINESS_MIN}%")
    df_filtered['OPM_clean'] = df_filtered['OPM_clean'].clip(lower=BUSINESS_MIN)

if opm_upper > BUSINESS_MAX:
    print(f"   ⚠️  Upper bound ({opm_upper:.2f}%) too extreme, capping at {BUSINESS_MAX}%")
    df_filtered['OPM_clean'] = df_filtered['OPM_clean'].clip(upper=BUSINESS_MAX)

# Final statistics
print("\n📊 CLEANING SUMMARY:")
print(f"   Original samples:     {len(df)}")
print(f"   After cleaning:       {len(df_filtered)}")
print(f"   Removed total:        {len(df) - len(df_filtered)} ({(len(df) - len(df_filtered))/len(df)*100:.2f}%)")
print(f"\n   Original OPM range:   [{df['OPM'].min():.2f}%, {df['OPM'].max():.2f}%]")
print(f"   Cleaned OPM range:    [{df_filtered['OPM_clean'].min():.2f}%, {df_filtered['OPM_clean'].max():.2f}%]")
print(f"\n   Original mean:        {df['OPM'].mean():.2f}%")
print(f"   Cleaned mean:         {df_filtered['OPM_clean'].mean():.2f}%")
print(f"   Original median:      {df['OPM'].median():.2f}%")
print(f"   Cleaned median:       {df_filtered['OPM_clean'].median():.2f}%")
print(f"\n   Original std:         {df['OPM'].std():.2f}%")
print(f"   Cleaned std:          {df_filtered['OPM_clean'].std():.2f}%")


STEP 3: Apply business rules...
   ⚠️  Lower bound (-61.06%) too extreme, capping at -50%

📊 CLEANING SUMMARY:
   Original samples:     4668
   After cleaning:       4167
   Removed total:        501 (10.73%)

   Original OPM range:   [-102800.00%, 3094.12%]
   Cleaned OPM range:    [-50.00%, 92.62%]

   Original mean:        -89.54%
   Cleaned mean:         13.96%
   Original median:      9.91%
   Cleaned median:       10.54%

   Original std:         2264.93%
   Cleaned std:          23.60%


In [16]:
df = df_filtered

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['OPM'].clip(-50, 100), bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[0].axvline(x=10, color='orange', linestyle='--', linewidth=2, label='10%')
axes[0].axvline(x=20, color='green', linestyle='--', linewidth=2, label='20%')
axes[0].set_xlabel('OPM (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('OPM Distribution - Before Cleaning\n(displayed -50% to 100%)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# After
axes[1].hist(df['OPM_clean'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[1].axvline(x=10, color='orange', linestyle='--', linewidth=2, label='10%')
axes[1].axvline(x=20, color='green', linestyle='--', linewidth=2, label='20%')
axes[1].set_xlabel('OPM (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('OPM Distribution - After Cleaning')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
os.makedirs(OUTPUT_PATH, exist_ok=True)
plt.savefig(f'{OUTPUT_PATH}01_opm_distribution_before_after.png', dpi=300, bbox_inches='tight')
print(f"\n   💾 Saved: {OUTPUT_PATH}01_opm_distribution_before_after.png")
plt.close()


   💾 Saved: output/opm_classification/01_opm_distribution_before_after.png
